In [59]:
import pandas as pd
import numpy as np
import kagglehub

path = kagglehub.dataset_download("mirichoi0218/insurance")
df = pd.read_csv('..' + path + '/insurance.csv')

quasi_ids = ['age', 'sex', 'region', 'children']
sensitive_column = 'charges'

for col in ['sex', 'region', 'smoker']:
    df[col] = df[col].astype('category')

def get_spans(df, partition):
    spans = {}
    for col in quasi_ids:
        if df[col].dtype.name == 'category':
            spans[col] = len(df[col][partition].unique())
        else:
            spans[col] = df[col][partition].max() - df[col][partition].min()
    return spans

def split(df, partition, column):
    dfp = df[column][partition]
    if df[column].dtype.name == 'category':
        values = dfp.unique()
        lv = set(values[:len(values)//2])
        rv = set(values[len(values)//2:])
        return dfp.index[dfp.isin(lv)], dfp.index[dfp.isin(rv)]
    else:
        median = dfp.median()
        return dfp.index[dfp < median], dfp.index[dfp >= median]

def mondrian(df, partition, k=10):
    if len(partition) < 2*k:
        return [partition]

    spans = get_spans(df, partition)
    for column in sorted(spans, key=spans.get, reverse=True):
        lp, rp = split(df, partition, column)
        if len(lp) >= k and len(rp) >= k:
            return mondrian(df, lp, k) + mondrian(df, rp, k)
    return [partition]

finished_partitions = mondrian(df, df.index, k=25)

rows = []
for part in finished_partitions:
    group = df.loc[part]
    agg_vals = {}
    for col in quasi_ids:
        if df[col].dtype.name == 'category':
            agg_vals[col] = ", ".join(group[col].unique())
        else:
            agg_vals[col] = f"{group[col].min()}-{group[col].max()}"
    agg_vals['charges_mean'] = group['charges'].mean()
    agg_vals['count'] = len(group)
    rows.append(agg_vals)

df = pd.DataFrame(rows)
df

,age,sex,region,children,charges_mean,count
0,18-19,female,"southwest, southeast",0-5,8375.520512,34
1,18-19,male,"southeast, southwest",0-2,8185.191000,37
2,18-19,female,"northeast, northwest",0-4,7739.256153,32
3,18-19,male,"northwest, northeast",0-3,9309.731907,34
4,20-21,"female, male","northwest, southeast",0-3,7907.115724,29
5,20-21,"female, male","northeast, southwest",0-5,7063.495699,28
6,22-23,"male, female","southwest, northeast",0-3,10483.766114,27
7,24-25,"male, female","northeast, southwest",0-5,11472.650964,28
8,22-25,male,"northwest, southeast",0-4,11272.569019,30
9,22-25,female,"southeast, northwest",0-3,9602.324910,27
